## NeuralProphet — Walmart Store Sales Forecasting (Google Colab)

**Setup (before Run all):**
1. **Runtime → Change runtime type** → T4 GPU optional (~30–45 min on CPU, faster on GPU)
2. **Secrets** (key icon in left sidebar):
   - `KAGGLE_API_TOKEN` — from kaggle.com/settings → API → Create token
   - `DAGSHUB_USER_TOKEN` — optional but recommended for MLflow
   - `WANDB_API_KEY` — full 40+ char secret (not the `wandb_v1_...` Key ID)
3. **First Run all:** cell **#0** installs packages then **restarts the runtime** — click **Run all** again when it comes back
4. **Second Run all:** cell **#0** prints `Deps ready` → training runs (~15 min with `FAST_RUN`)

**Why Colab not Kaggle?** Kaggle Python 3.12 has unfixable `pytorch-lightning` / `lightning-fabric` conflicts for NeuralProphet.

**Quick demo for lecturer:** keep `FAST_RUN = True` in cell **#3** (~15 min on CPU).

**Full run:** set `FAST_RUN = False`, `LONG_RUN = False` (~30–45 min).

In [ ]:
#0 — install NeuralProphet (run FIRST; auto-restarts once, then Run all again)
import os
import subprocess
import sys

FLAG = '/content/.neuralprophet_deps_v8'


def _pip(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


if os.path.exists(FLAG):
    print('Deps ready — continuing')
else:
    subprocess.call([
        sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
        'prophet', 'tensorflow', 'tensorflow-datasets',
    ])
    _pip(
        'neuralprophet==1.0.0rc10',
        'pytorch-lightning==2.3.3',
        'lightning-fabric==2.3.3',
        'dagshub', 'mlflow', 'wandb',
    )
    subprocess.check_call([
        sys.executable, '-c',
        'import pytorch_lightning as pl; import lightning_fabric as lf; '
        'from neuralprophet import NeuralProphet; '
        'print(f"OK PL={pl.__version__} fabric={lf.__version__}")',
    ])
    open(FLAG, 'w').close()
    print('Install OK — restarting runtime. When Colab reloads, click Run all again.')
    os.kill(os.getpid(), 9)


In [ ]:
#1 — download + load data (Colab via Kaggle API)
import os
import numpy as np
import pandas as pd

COMPETITION_SLUG = 'walmart-recruiting-store-sales-forecasting'
DATA_DIR = f'/content/kaggle/input/competitions/{COMPETITION_SLUG}'


def read_competition_csv(data_dir, stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f'Missing {stem}.csv or {stem}.csv.zip in {data_dir}')


train_zip = os.path.join(DATA_DIR, 'train.csv')
if not os.path.exists(train_zip):
    print('Downloading competition data via Kaggle API...')
    try:
        from google.colab import userdata
        kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    except Exception as e:
        raise RuntimeError(
            'Add KAGGLE_API_TOKEN to Colab Secrets (key icon → Add secret). '
            'Get token from kaggle.com/settings → API → Create New Token.'
        ) from e
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
        f.write(kaggle_token)
    os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
    get_ipython().system('pip install -q kaggle')
    os.makedirs(DATA_DIR, exist_ok=True)
    get_ipython().system(f'kaggle competitions download -c {COMPETITION_SLUG} -p {DATA_DIR}')
    zip_path = os.path.join(DATA_DIR, f'{COMPETITION_SLUG}.zip')
    if os.path.exists(zip_path):
        get_ipython().system(f'unzip -q {zip_path} -d {DATA_DIR}')
        os.remove(zip_path)

print(f'Using data directory: {DATA_DIR}')
print('Files:', sorted(os.listdir(DATA_DIR)))

train = read_competition_csv(DATA_DIR, 'train')
test = read_competition_csv(DATA_DIR, 'test')
stores = read_competition_csv(DATA_DIR, 'stores')
features = read_competition_csv(DATA_DIR, 'features')

train['Date'] = pd.to_datetime(train['Date'])
test['Date'] = pd.to_datetime(test['Date'])
features['Date'] = pd.to_datetime(features['Date'])

train = train.merge(stores, on='Store', how='left')
train = train.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')
test = test.merge(stores, on='Store', how='left')
test = test.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
train[md_cols] = train[md_cols].fillna(0)
test[md_cols] = test[md_cols].fillna(0)

for col in ['CPI', 'Unemployment']:
    train[col] = train.groupby('Store')[col].transform(lambda x: x.ffill().bfill())
    test[col] = test.groupby('Store')[col].transform(lambda x: x.ffill().bfill())

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)
train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(f'Data loaded | Train: {train.shape} | series: {train.groupby(["Store","Dept"]).ngroups}')

In [ ]:
#2 — setup (deps installed in cell #0)
NOTEBOOK_VERSION = 'neuralprophet_v8_colab_restart'

import json, logging, os, warnings
from datetime import datetime
from pathlib import Path

import pandas as pd

# pandas 2.x removed Series.view — neuralprophet 1.0.0rc10 still calls it internally
if not hasattr(pd.Series, 'view'):
    pd.Series.view = lambda self, *args, **kwargs: self.to_numpy().view(*args, **kwargs)

import matplotlib.pyplot as plt
import mlflow
import dagshub
from neuralprophet import NeuralProphet, set_log_level

warnings.filterwarnings('ignore')
set_log_level('ERROR')
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

WORK_DIR = Path('/content')

PROGRESS_LOG = WORK_DIR / 'neuralprophet_progress.log'
RUN_COMPLETE = WORK_DIR / 'RUN_COMPLETE.txt'
if RUN_COMPLETE.exists():
    RUN_COMPLETE.unlink()

def log_progress(msg: str):
    line = f"[{datetime.utcnow().isoformat()}Z] {msg}"
    print(line)
    with open(PROGRESS_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def _get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_dagshub_token = _get_secret('DAGSHUB_USER_TOKEN')
if _dagshub_token:
    dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True, token=_dagshub_token)
    log_progress('DagsHub auth: token')
else:
    dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)
    log_progress('DagsHub auth: oauth')

mlflow.set_experiment('NeuralProphet_Training')

USE_WANDB = False
try:
    import wandb
    _wandb_key = _get_secret('WANDB_API_KEY')
    if _wandb_key:
        wandb.login(key=_wandb_key)
        USE_WANDB = True
        log_progress('W&B enabled')
    else:
        log_progress('W&B disabled: no WANDB_API_KEY')
except Exception as e:
    log_progress(f'W&B disabled: {e}')

log_progress(f'Notebook version: {NOTEBOOK_VERSION} | WORK_DIR={WORK_DIR}')
log_progress('Setup complete')

NeuralProphet uses `ds` + `y` like Prophet, plus **`n_lags`**, **`epochs`**, and **`add_future_regressor`** for holiday/economic features.

In [ ]:
#3
FAST_RUN = True   # True = ~15 min lecturer demo | False = full experiment grid
LONG_RUN = False  # only used when FAST_RUN=False

NP_EPOCHS = 15 if FAST_RUN else (80 if LONG_RUN else 40)
NP_N_LAGS = 26 if FAST_RUN else 52

def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

SPLIT_DATE = train['Date'].max() - pd.Timedelta(weeks=8)
train_set = train[train['Date'] <= SPLIT_DATE].copy()
val_set = train[train['Date'] > SPLIT_DATE].copy()

log_progress(f'FAST_RUN={FAST_RUN} | LONG_RUN={LONG_RUN} | epochs={NP_EPOCHS} | n_lags={NP_N_LAGS}')
log_progress(f'Train: {train_set.Date.min().date()} -> {train_set.Date.max().date()}')
log_progress(f'Val  : {val_set.Date.min().date()} -> {val_set.Date.max().date()} ({val_set.Date.nunique()} weeks)')

In [ ]:
#4
def to_np_df(df, store, dept):
    part = df[(df['Store'] == store) & (df['Dept'] == dept)].copy()
    out = part.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    out['IsHoliday'] = out['IsHoliday'].fillna(0).astype(float)
    return out[['ds', 'y', 'IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']]

store, dept = 1, 1
train_np = to_np_df(train_set, store, dept)
val_np = to_np_df(val_set, store, dept)

with mlflow.start_run(run_name='NeuralProphet_Cleaning'):
    mlflow.log_params({'validation_weeks': 8, 'store': store, 'dept': dept, 'fast_run': FAST_RUN, 'long_run': LONG_RUN})
log_progress(f'Single series: Store {store} Dept {dept} | train={len(train_np)} val={len(val_np)}')

RUN_METRICS = {}

def fit_predict_neuralprophet(train_df, val_df, params, regressors=None, use_holidays=False, run_name=''):
    regressors = regressors or []
    m = NeuralProphet(**params)
    if use_holidays:
        m = m.add_country_holidays('US')
    for reg in regressors:
        m = m.add_future_regressor(reg, normalize='auto')

    fit_cols = ['ds', 'y'] + regressors
    history = m.fit(train_df[fit_cols], freq='W')

    combined = pd.concat([train_df[fit_cols], val_df[fit_cols]], ignore_index=True)
    forecast = m.predict(combined)
    fc_val = forecast[forecast['ds'].isin(val_df['ds'])][['ds', 'yhat1']].sort_values('ds')
    merged = val_df[['ds', 'y', 'IsHoliday']].merge(fc_val, on='ds', how='inner').sort_values('ds')
    if len(merged) == 0:
        raise ValueError(f'No overlapping validation dates for {run_name or "run"}')
    if len(merged) != len(val_df):
        log_progress(f'{run_name}: scoring {len(merged)}/{len(val_df)} val weeks (date alignment)')
    preds = np.clip(merged['yhat1'].values, 0, None)
    score = wmae(merged['y'].values, preds, merged['IsHoliday'].values)

    if run_name:
        RUN_METRICS[run_name] = {'val_wmae': score, 'n_train_epochs': len(history) if history is not None else 0}
        log_progress(f'{run_name} WMAE={score:,.2f}')
    return m, preds, score, history

BASE_PARAMS = {
    'n_lags': NP_N_LAGS,
    'n_forecasts': 1,
    'yearly_seasonality': True,
    'weekly_seasonality': True,
    'daily_seasonality': False,
    'learning_rate': 0.01,
    'epochs': NP_EPOCHS,
    'batch_size': 64,
}
log_progress('fit_predict_neuralprophet + BASE_PARAMS ready')

In [ ]:
#5 — baseline run
m_base, preds_base, score_base, _ = fit_predict_neuralprophet(
    train_np, val_np, BASE_PARAMS, regressors=None, use_holidays=False, run_name='NeuralProphet_Baseline'
)

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
ax.plot(train_np['ds'], train_np['y'], color='#58a6ff', label='Train')
ax.plot(val_np['ds'], val_np['y'], color='#3fb950', label='Actual val')
ax.plot(val_np['ds'], preds_base, color='#f78166', linestyle='--', label='NeuralProphet')
ax.axvline(SPLIT_DATE, color='white', alpha=0.35, linestyle=':')
ax.set_title(f'NeuralProphet baseline WMAE={score_base:,.0f}', color='white')
ax.legend()
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'neuralprophet_baseline.png'), dpi=130, facecolor='#0f1117')
plt.show()

with mlflow.start_run(run_name='NeuralProphet_Baseline'):
    mlflow.log_params(BASE_PARAMS)
    mlflow.log_metric('val_wmae', score_base)
    mlflow.log_artifact(str(WORK_DIR / 'neuralprophet_baseline.png'), 'plots')

In [ ]:
#6 — holidays
assert 'fit_predict_neuralprophet' in globals(), 'Re-run cell #4 first (defines fit_predict_neuralprophet)'
_, _, score_hol, _ = fit_predict_neuralprophet(
    train_np, val_np, BASE_PARAMS, regressors=None, use_holidays=True, run_name='NeuralProphet_Holidays'
)

with mlflow.start_run(run_name='NeuralProphet_Holidays'):
    mlflow.log_params({**BASE_PARAMS, 'country_holidays': 'US'})
    mlflow.log_metric('val_wmae', score_hol)

In [ ]:
#7
REGRESSOR_SETS = (
    {'NP_Regressor_Full': ['IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']}
    if FAST_RUN else
    {
        'NP_Regressor_IsHoliday': ['IsHoliday'],
        'NP_Regressor_Economic': ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment'],
        'NP_Regressor_Full': ['IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment'],
    }
)

regressor_results = {'NeuralProphet_Baseline': score_base, 'NeuralProphet_Holidays': score_hol}

for run_name, regs in REGRESSOR_SETS.items():
    _, _, score, _ = fit_predict_neuralprophet(
        train_np, val_np, BASE_PARAMS, regressors=regs, use_holidays=True, run_name=run_name
    )
    regressor_results[run_name] = score
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({**BASE_PARAMS, 'regressors': regs, 'country_holidays': 'US'})
        mlflow.log_metric('val_wmae', score)

In [ ]:
#8
best_regs_name = min(REGRESSOR_SETS, key=lambda k: regressor_results[k])
best_regressors = REGRESSOR_SETS[best_regs_name]

HYPERPARAM_GRID = {} if FAST_RUN else {
    'NP_HP_MoreLags': {'n_lags': 104, 'epochs': 60 if LONG_RUN else 40},
    'NP_HP_LowLR': {'learning_rate': 0.001, 'epochs': 60 if LONG_RUN else 50},
    'NP_HP_DeepAR': {'n_lags': 52, 'n_forecasts': 4, 'epochs': 60 if LONG_RUN else 45},
}

hyper_results = {}
for run_name, extra in HYPERPARAM_GRID.items():
    params = {**BASE_PARAMS, **extra}
    _, _, score, _ = fit_predict_neuralprophet(
        train_np, val_np, params, regressors=best_regressors, use_holidays=True, run_name=run_name
    )
    hyper_results[run_name] = score
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({**params, 'regressors': best_regressors})
        mlflow.log_metric('val_wmae', score)

In [ ]:
#9
def evaluate_np_subset(params, regressors, use_holidays=True, n_series=50):
    np.random.seed(42)
    all_pairs = train.groupby(['Store', 'Dept']).size()
    valid_pairs = all_pairs[all_pairs >= 80].index.tolist()
    n = min(n_series, len(valid_pairs))
    sample_pairs = [valid_pairs[i] for i in np.random.choice(len(valid_pairs), size=n, replace=False)]

    rows, n_failed = [], 0
    for store_i, dept_i in sample_pairs:
        tr = to_np_df(train_set, store_i, dept_i)
        vl = to_np_df(val_set, store_i, dept_i)
        if len(tr) < 80 or len(vl) == 0:
            n_failed += 1
            continue
        try:
            _, preds, _, _ = fit_predict_neuralprophet(tr, vl, params, regressors=regressors, use_holidays=use_holidays)
            for idx, (_, row) in enumerate(vl.iterrows()):
                if idx < len(preds):
                    rows.append({'Actual': row['y'], 'Predicted': preds[idx], 'IsHoliday': row['IsHoliday']})
        except Exception:
            n_failed += 1

    if not rows:
        return None, n_failed
    pred_df = pd.DataFrame(rows)
    return wmae(pred_df['Actual'].values, pred_df['Predicted'].values, pred_df['IsHoliday'].values), n_failed

single_series_candidates = {**regressor_results, **hyper_results}
best_single_name = min(single_series_candidates, key=single_series_candidates.get)

if best_single_name in HYPERPARAM_GRID:
    champion_params = {**BASE_PARAMS, **HYPERPARAM_GRID[best_single_name]}
    champion_regressors = best_regressors
elif best_single_name in REGRESSOR_SETS:
    champion_params = BASE_PARAMS.copy()
    champion_regressors = REGRESSOR_SETS[best_single_name]
elif best_single_name == 'NeuralProphet_Holidays':
    champion_params = BASE_PARAMS.copy()
    champion_regressors = []
else:
    champion_params = BASE_PARAMS.copy()
    champion_regressors = []

if FAST_RUN:
    subset_score, subset_failed = None, 0
    log_progress('Subset50 skipped (FAST_RUN)')
else:
    subset_score, subset_failed = evaluate_np_subset(champion_params, champion_regressors, use_holidays=True)
    RUN_METRICS['NeuralProphet_Subset50'] = {'val_wmae': subset_score, 'n_failed': subset_failed}
    log_progress(f'Subset50 WMAE={subset_score:,.2f} | failed={subset_failed}')
    with mlflow.start_run(run_name='NeuralProphet_Subset50'):
        mlflow.log_params({**champion_params, 'regressors': champion_regressors})
        mlflow.log_metric('val_wmae', subset_score)
        mlflow.log_metric('n_failed', subset_failed)
        mlflow.log_param('source_config', best_single_name)

In [ ]:
#10
all_scores = {**single_series_candidates}
if subset_score is not None:
    all_scores['NeuralProphet_Subset50'] = subset_score
all_scores = {k: v for k, v in all_scores.items() if v is not None}

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
names = list(all_scores.keys())
scores = list(all_scores.values())
colors = ['#8b949e'] * len(names)
colors[int(np.argmin(scores))] = '#f78166'
bars = ax.bar(names, scores, color=colors, width=0.5)
ax.bar_label(bars, labels=[f'{v:,.1f}' for v in scores], color='white', fontsize=9)
ax.set_title('NeuralProphet runs (WMAE, lower is better)', color='white')
ax.tick_params(colors='#8b949e', axis='x', rotation=25)
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'neuralprophet_all_runs.png'), dpi=130, facecolor='#0f1117')
plt.show()

champion_model, champion_preds, champion_val_score, _ = fit_predict_neuralprophet(
    train_np, val_np, champion_params, regressors=champion_regressors, use_holidays=True, run_name='NeuralProphet_Best'
)

model_path = WORK_DIR / 'neuralprophet_best.pt'
import torch
torch.save({'config': champion_params, 'regressors': champion_regressors, 'best_config': best_single_name}, model_path)

with mlflow.start_run(run_name='NeuralProphet_Best'):
    mlflow.log_params({**champion_params, 'regressors': champion_regressors, 'best_config': best_single_name})
    mlflow.log_metric('val_wmae_single', champion_val_score)
    if subset_score is not None:
        mlflow.log_metric('val_wmae_subset50', subset_score)
    mlflow.log_artifact(str(model_path), 'model')
    mlflow.log_artifact(str(WORK_DIR / 'neuralprophet_all_runs.png'), 'plots')
    try:
        np_save_dir = WORK_DIR / 'neuralprophet_best_np'
        champion_model.save(str(np_save_dir))
        mlflow.log_artifact(str(np_save_dir), 'neuralprophet_native')
    except Exception as e:
        log_progress(f'Native NP save skipped: {e}')

subset_msg = f'{subset_score:,.2f}' if subset_score is not None else 'skipped (FAST_RUN)'
log_progress(f'Best: {best_single_name} | single WMAE={champion_val_score:,.2f} | subset50={subset_msg}')

In [ ]:
#11
summary = {
    'status': 'COMPLETE',
    'finished_at_utc': datetime.utcnow().isoformat() + 'Z',
    'notebook_version': NOTEBOOK_VERSION,
    'best_config': best_single_name,
    'champion_val_wmae': float(champion_val_score),
    'subset50_wmae': float(subset_score) if subset_score is not None else None,
    'all_scores': {k: float(v) for k, v in all_scores.items()},
}

with open(RUN_COMPLETE, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

log_progress('RUN COMPLETE')
print(json.dumps(summary, indent=2))

In [ ]:
#12 — W&B logging (logs in directly; works even if cell #2 ran before secret was added)
import wandb

_wandb_ok = False
try:
    _wandb_key = _get_secret('WANDB_API_KEY')
    if not _wandb_key or len(_wandb_key) < 40:
        raise RuntimeError('WANDB_API_KEY missing or too short (use full secret, not Key ID)')
    wandb.login(key=_wandb_key)
    _wandb_ok = True
except Exception as e:
    log_progress(f'W&B skipped: {e}')

if _wandb_ok:
    for run_name, metrics in RUN_METRICS.items():
        wb = wandb.init(project='walmart-forecasting', name=run_name, reinit=True)
        wandb.log(metrics)
        wb.finish()
        log_progress(f'W&B logged -> {run_name}')

    summary_wb = wandb.init(project='walmart-forecasting', name='NeuralProphet_Summary', reinit=True)
    wandb.log({f'wmae/{k}': v for k, v in all_scores.items()})
    summary_metrics = {'best_config': best_single_name, 'champion_val_wmae': champion_val_score}
    if subset_score is not None:
        summary_metrics['subset50_wmae'] = subset_score
    wandb.log(summary_metrics)
    chart = WORK_DIR / 'neuralprophet_all_runs.png'
    if chart.exists():
        wandb.log({'comparison_chart': wandb.Image(str(chart))})
    summary_wb.finish()
    log_progress('W&B summary -> NeuralProphet_Summary')
    print('W&B: https://wandb.ai — project walmart-forecasting')